In [1]:
# 05_judge_evaluation.ipynb -- long-format judge scoring with judge-condition ablation + anchor-recovery + inter-judge probes

# === Shared configuration (self-contained: one code cell per notebook) ===
from __future__ import annotations
import os, sys, json, math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Resolve the repository root (the folder that holds both 'results' and 'notebooks').
def _find_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "results").is_dir() and (cand / "notebooks").is_dir():
            return cand
    return start.resolve().parent if start.resolve().name == "notebooks" else start.resolve()

ROOT = _find_root(Path.cwd())
DATA, RAW, PROC = ROOT / "data", ROOT / "data" / "raw", ROOT / "data" / "processed"
RESULTS, FIGS, TABS = ROOT / "results", ROOT / "results" / "figures", ROOT / "results" / "tables"
for d in (DATA, RAW, PROC, RESULTS, FIGS, TABS):
    d.mkdir(parents=True, exist_ok=True)

# Load environment variables (.env is git-ignored; never commit secrets).
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except Exception:
    pass
LLM_MODELS = {
    "weak":   os.getenv("LLM_MODEL_WEAK", ""),
    "mid":    os.getenv("LLM_MODEL_MID", ""),
    "strong": os.getenv("LLM_MODEL_STRONG", ""),
}
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

# Plotting convention: seaborn, grayscale, no caption, 600 dpi, saved as PNG and PDF.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(context="paper", style="ticks")
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 600, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.grid": True, "grid.alpha": 0.3, "image.cmap": "Greys",
})

def gray_palette(n: int):
    # Evenly spaced grays (mid -> dark) for categorical series.
    return sns.color_palette("Greys", max(n + 2, 3))[2:n + 2]

def save_fig(fig, name: str):
    # Save a figure as PNG and PDF at 600 dpi; no caption is added here.
    for ext in ("png", "pdf"):
        fig.savefig(FIGS / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

def save_table(df: pd.DataFrame, name: str, index: bool = False):
    # Persist a results table as CSV and (best-effort) LaTeX for the manuscript.
    df.to_csv(TABS / f"{name}.csv", index=index)
    try:
        (TABS / f"{name}.tex").write_text(df.to_latex(index=index))
    except Exception:
        pass

# Dataframe I/O that needs no parquet engine: use Parquet if one is installed, else CSV.
def _parquet_ok():
    for mod in ("pyarrow", "fastparquet"):
        try:
            __import__(mod); return True
        except Exception:
            pass
    return False
PARQUET = _parquet_ok()
def write_df(df: pd.DataFrame, stem: Path):
    # 'stem' is a path without extension; returns the path actually written.
    if PARQUET:
        p = stem.with_suffix(".parquet"); df.to_parquet(p); return p
    p = stem.with_suffix(".csv"); df.to_csv(p, index=False); return p
def df_exists(stem: Path) -> bool:
    return stem.with_suffix(".parquet").exists() or stem.with_suffix(".csv").exists()
def read_df(stem: Path) -> pd.DataFrame:
    pq, cv = stem.with_suffix(".parquet"), stem.with_suffix(".csv")
    if pq.exists(): return pd.read_parquet(pq)
    if cv.exists(): return pd.read_csv(cv)
    raise FileNotFoundError(f"{stem}.parquet/.csv")

# Fine-grained rating scale (low -> high); used to map labels to an ordinal rank.
RATING_SCALE = ["D","C","CC","CCC-","CCC","CCC+","B-","B","B+","BB-","BB","BB+",
                "BBB-","BBB","BBB+","A-","A","A+","AA-","AA","AA+","AAA"]
RANK = {g: i for i, g in enumerate(RATING_SCALE)}
def to_rank(label):
    # Map a rating label to an ordinal rank; returns np.nan if unmatched.
    if not isinstance(label, str):
        return np.nan
    s = label.strip().upper().replace(" ", "")
    return RANK.get(s, np.nan)

# --- 05: judge scoring in LONG format with a judge-condition ablation and two validation probes ---
# Output columns add: gen_condition (from 04), judge_model (primary/secondary), judge_condition
# (informed = judge told the notch gap; blind = not told). A second judge scores a sample for
# inter-judge agreement, and a known-anchor-recovery probe rates rating extremes from financials only.
INTERJUDGE_SAMPLE = 200   # rows scored by the secondary judge (for agreement)
N_ANCHORS = 40            # extreme-rating anchors for the recovery probe
src = PROC / "perturbations"
if not OPENAI_API_KEY:
    print("[skip] no OPENAI_API_KEY in .env")
elif not df_exists(src):
    print("[skip] run 04_perturbation first (no perturbations).")
else:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    models_avail = [m for m in (LLM_MODELS.get("strong"), LLM_MODELS.get("mid"),
                                LLM_MODELS.get("weak")) if m]
    JUDGE_PRIMARY = models_avail[0] if models_avail else ""
    JUDGE_SECONDARY = next((m for m in models_avail if m != JUDGE_PRIMARY), None)

    def ask_json(model, system, user, max_tokens=400):
        for extra in (dict(temperature=0.0, max_tokens=max_tokens),
                      dict(max_completion_tokens=max_tokens), dict()):
            try:
                r = client.chat.completions.create(
                    model=model,
                    messages=[{"role": "system", "content": system},
                              {"role": "user", "content": user}], **extra)
                return r.choices[0].message.content
            except Exception:
                continue
        return ""

    def parse(js, keys):
        try:
            d = json.loads(js)
        except Exception:
            d = {}
        return {k: d.get(k, np.nan) for k in keys}

    scored_src = PROC / "override_subset"
    scored_src = scored_src if df_exists(scored_src) else (PROC / "corpus_scored")
    delta_map = {}
    if df_exists(scored_src):
        sdf0 = read_df(scored_src).reset_index(drop=True)
        if "delta" in sdf0.columns:
            delta_map = {i: int(sdf0.iloc[i]["delta"]) for i in range(len(sdf0))}

    def row_delta(r):
        d = r.get("delta")
        try:
            d = int(d)
        except Exception:
            d = None
        if d is None:
            d = delta_map.get(int(r["case_id"]))
        return d

    def notch_note(d):
        if d is None or d == 0:
            return "NOTCH_GAP: not available (treat as a standalone rating)."
        direction = "ABOVE" if d > 0 else "BELOW"
        return (f"NOTCH_GAP: the agency rating is {abs(d)} notch(es) {direction} what the standalone "
                f"financials imply, so a good rationale should present a qualitative/contextual "
                f"justification whose direction matches this gap.")

    RUBRIC = ("You evaluate a credit-rating output. The issuer's agency rating differs from its "
              "standalone financials by a stated NOTCH_GAP, so a good rationale presents a "
              "qualitative/contextual justification (parent/group/government support, industry, "
              "governance) whose direction matches the gap. Score each 1-5 (integers): "
              "grade_accuracy, grade_rationale_consistency, qualitative_adequacy (5 = clear and "
              "consistent with the gap's direction, 3 = present but generic, 1 = absent or contradicts "
              "the gap), factual_faithfulness, explainability. Return strict JSON with those five keys.")
    AXES = ("grade_accuracy", "grade_rationale_consistency", "qualitative_adequacy",
            "factual_faithfulness", "explainability")
    FAITH = ("Given FINANCIAL_DATA and ANSWER, return JSON {\"faithfulness\": <0-1>} = the fraction "
             "of the answer's factual/quantitative claims entailed by FINANCIAL_DATA. Ignore whether "
             "the grade is wise and ignore unverifiable qualitative statements.")

    def rubric_user(fin, note, ans):
        return f"FINANCIAL_PROFILE:\n{fin}\n{note}\nOUTPUT:\n{ans}"

    pert = read_df(src)
    rng = np.random.default_rng(SEED)
    n = len(pert)
    sample_ids = set(rng.choice(n, min(INTERJUDGE_SAMPLE, n), replace=False).tolist())
    keep = ("case_id", "model_tag", "gen_condition", "ptype", "level", "delta")
    out = []
    for idx, (_, r) in enumerate(pert.iterrows()):
        ans = f"Rating: {r['rating']}\nRationale: {r['rationale']}"
        fin = str(r.get("fin_profile", "")) or "(none provided)"
        d = row_delta(r)
        faith = parse(ask_json(JUDGE_PRIMARY, FAITH, f"FINANCIAL_DATA:\n{fin}\nANSWER:\n{ans}"),
                      ["faithfulness"]).get("faithfulness", np.nan)
        base = {k: r.get(k) for k in keep}
        for jc in ("informed", "blind"):
            note = notch_note(d) if jc == "informed" else \
                "NOTCH_GAP: not available (treat as a standalone rating)."
            sc = parse(ask_json(JUDGE_PRIMARY, RUBRIC, rubric_user(fin, note, ans)), AXES)
            out.append({**base, "judge_model": "primary", "judge_condition": jc,
                        **sc, "ragas_faithfulness": faith})
        if JUDGE_SECONDARY and idx in sample_ids:
            sc2 = parse(ask_json(JUDGE_SECONDARY, RUBRIC, rubric_user(fin, notch_note(d), ans)), AXES)
            out.append({**base, "judge_model": "secondary", "judge_condition": "informed",
                        **sc2, "ragas_faithfulness": faith})
    scores = pd.DataFrame(out)
    write_df(scores, PROC / "judge_scores")
    print("wrote judge scores (long):", len(scores), "rows")

    # Probe (ii): known-anchor recovery -- judge rates rating extremes from the profile alone.
    anchor_rows = []
    if df_exists(scored_src):
        sdf = read_df(scored_src).reset_index(drop=True)
        feat = json.loads((PROC / "feature_columns.json").read_text())
        if "rating_rank" in sdf.columns and len(sdf):
            order = sdf["rating_rank"].sort_values()
            k = max(1, min(N_ANCHORS // 2, len(order) // 2))
            anchors = list(order.index[:k]) + list(order.index[-k:])
            ANCHOR_SYS = ("You are a credit analyst. From the financial profile ALONE, assign an "
                          "issuer rating on the scale AAA (best) to D (default). "
                          "Return strict JSON {\"rating\": <string>}.")
            for ci in anchors:
                row = sdf.iloc[ci]
                profile = "; ".join(f"{c}={row[c]:.4g}" for c in feat if pd.notna(row[c]))
                js = ask_json(JUDGE_PRIMARY, ANCHOR_SYS, f"Financial profile: {profile}\nReturn JSON only.")
                try:
                    pr = json.loads(js).get("rating", "")
                except Exception:
                    pr = ""
                jr = to_rank(pr)
                ar = int(row["rating_rank"])
                anchor_rows.append(dict(case_id=int(ci), agency_rank=ar,
                                        judge_rank=(np.nan if np.isnan(jr) else int(jr)),
                                        abs_err=(np.nan if np.isnan(jr) else abs(int(jr) - ar))))
        write_df(pd.DataFrame(anchor_rows), PROC / "anchor_scores")
        print("wrote anchor_scores:", len(anchor_rows), "rows")


wrote judge scores (long): 5000 rows
wrote anchor_scores: 40 rows
